In [1]:
import os
import sys
import numpy as np
import torch

sys.path.append(os.path.abspath('..'))
from src.lstm_encoder import HybridCNNBiLSTMEncoder, StandaloneBiLSTMEncoder

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Testing Phase 7 Encoder on: {device}')

# -------------------------------------------------------------
# Test Case 1: Standalone BiLSTM Verification
# Input: (Batch=32, Seq_Len=30, Feature_Dim=64)
# -------------------------------------------------------------
dummy_cnn_out = torch.randn(32, 30, 64).to(device)
bilstm_unit = StandaloneBiLSTMEncoder(
    in_dim=64, hidden_dim=64, num_layers=2, pool_strategy='mean'
).to(device)

rep, (h_n, c_n) = bilstm_unit(dummy_cnn_out)
print('\n--- Test Case 1: Standalone BiLSTM Unit Test ---')
print(f'Input Shape:        {tuple(dummy_cnn_out.shape)}')
print(f'Output Embed Shape: {tuple(rep.shape)} (Expected: (32, 128))')
print(f'Hidden State Shape: {tuple(h_n.shape)} (Expected: (4, 32, 64))')
assert rep.shape == (32, 128), 'Standalone BiLSTM output mismatch!'

# -------------------------------------------------------------
# Test Case 2: Full Connected Hybrid Encoder (FDI Feeder Shape)
# Input: (Batch=32, Seq_Len=30, In_Channels=12)
# -------------------------------------------------------------
fdi_batch = torch.randn(32, 30, 12).to(device)
hybrid_fdi = HybridCNNBiLSTMEncoder(
    in_channels=12,
    cnn_hidden_dims=(32, 64),
    lstm_hidden_dim=64,
    pool_strategy='mean',
).to(device)

z_fdi = hybrid_fdi(fdi_batch)
print('\n--- Test Case 2: Connected Hybrid Encoder (FDI Feeder Data) ---')
print(f'Input Raw Telemetry:    {tuple(fdi_batch.shape)}')
print(f'Final Latent Embedding: {tuple(z_fdi.shape)} (Expected: (32, 128))')
assert z_fdi.shape == (32, 128), 'Hybrid FDI output shape mismatch!'

# -------------------------------------------------------------
# Test Case 3: Full Connected Hybrid Encoder (LCL Pretraining Shape)
# Input: (Batch=16, Seq_Len=48, In_Channels=25)
# -------------------------------------------------------------
lcl_batch = torch.randn(16, 48, 25).to(device)
hybrid_lcl = HybridCNNBiLSTMEncoder(
    in_channels=25,
    cnn_hidden_dims=(64, 128),
    lstm_hidden_dim=128,
    pool_strategy='mean',
).to(device)

z_lcl = hybrid_lcl(lcl_batch)
print('\n--- Test Case 3: Connected Hybrid Encoder (LCL Pretraining Data) ---')
print(f'Input Load Matrix:      {tuple(lcl_batch.shape)}')
print(f'Final Latent Embedding: {tuple(z_lcl.shape)} (Expected: (16, 256))')
assert z_lcl.shape == (16, 256), 'Hybrid LCL output shape mismatch!'

# -------------------------------------------------------------
# Test Case 4: End-to-End Gradient Flow (Backpropagation)
# -------------------------------------------------------------
loss = z_lcl.sum()
loss.backward()

cnn_grads = [
    p.grad.norm().item()
    for p in hybrid_lcl.cnn.parameters()
    if p.grad is not None
]
lstm_grads = [
    p.grad.norm().item()
    for p in hybrid_lcl.bilstm.parameters()
    if p.grad is not None
]

print('\n--- Test Case 4: Backpropagation & Joint Gradient Flow ---')
print(
    'CNN front-end receiving gradients: '
    f' {all(g > 0.0 for g in cnn_grads)} (Mean norm: {np.mean(cnn_grads):.4f})'
)
print(
    'BiLSTM back-end receiving gradients:'
    f' {all(g > 0.0 for g in lstm_grads)} (Mean norm: {np.mean(lstm_grads):.4f})'
)
assert all(g > 0.0 for g in cnn_grads) and all(
    g > 0.0 for g in lstm_grads
), 'Gradient flow interrupted between CNN and BiLSTM!'

print('\n[SUCCESS] Phase 7 BiLSTM and Connected Hybrid Encoder Verified.')

Testing Phase 7 Encoder on: cuda

--- Test Case 1: Standalone BiLSTM Unit Test ---
Input Shape:        (32, 30, 64)
Output Embed Shape: (32, 128) (Expected: (32, 128))
Hidden State Shape: (4, 32, 64) (Expected: (4, 32, 64))

--- Test Case 2: Connected Hybrid Encoder (FDI Feeder Data) ---
Input Raw Telemetry:    (32, 30, 12)
Final Latent Embedding: (32, 128) (Expected: (32, 128))

--- Test Case 3: Connected Hybrid Encoder (LCL Pretraining Data) ---
Input Load Matrix:      (16, 48, 25)
Final Latent Embedding: (16, 256) (Expected: (16, 256))

--- Test Case 4: Backpropagation & Joint Gradient Flow ---
CNN front-end receiving gradients:  True (Mean norm: 9.2927)
BiLSTM back-end receiving gradients: True (Mean norm: 65.1107)

[SUCCESS] Phase 7 BiLSTM and Connected Hybrid Encoder Verified.
